# 07 — Optuna hyperparameter tuning

Tune ExtraTreesRegressor, CatBoostRegressor, and SVR using training compounds only and five-fold scaffold-grouped cross-validation. Every trial uses the same ordered, fixed 250-feature list selected in Stage 05; there is no feature cleaning, ranking, or reselection within these folds. SVR is tuned with a StandardScaler pipeline; scaling is fit within each training fold.

Optuna minimizes the mean of the five validation-fold RMSE values. For each trial, the notebook records fold RMSE, MSE, MAE, and R²; the best trial’s fold metrics are summarized as mean ± sample SD. The corresponding pooled out-of-fold metrics are also reported. These results are taken from the best trial’s already completed folds, so no second CV refit is run after tuning.

Because the fixed 250-feature list was selected using the full training partition before these folds were formed, these CV metrics are conditional on that list and can be optimistic. The locked test set remains untouched and is used once for final external evaluation. No imputation is used.

Optuna resumes saved studies. Set `optuna_trials` in `config.json` before the first run. The updated feature-list and script hashes create a new study rather than resuming the earlier fold-specific feature-selection study.

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print("Project root:", PROJECT_ROOT)


Project root: c:\Users\USER\Documents\E\Code


In [ ]:
from src.optuna_tuning import run
tuning_results = run()
tuning_results

Optuna: ExtraTreesRegressor, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-12 16:51:55,462] A new study created in RDB with name: brd4_f081bd1fe370_ExtraTreesRegressor
[I 2026-11-12 16:55:53,835] Trial 0 finished with value: 0.6086277962795018 and parameters: {'n_estimators': 500, 'max_features': 0.9704285838459497, 'min_samples_split': 15, 'min_samples_leaf': 5, 'max_depth': 30, 'bootstrap': False}. Best is trial 0 with value: 0.6086277962795018.
[I 2026-11-12 17:04:55,837] Trial 1 finished with value: 0.5978150809650088 and parameters: {'n_estimators': 900, 'max_features': 0.899465584480253, 'min_samples_split': 6, 'min_samples_leaf': 2, 'max_depth': 20, 'bootstrap': False}. Best is trial 1 with value: 0.5978150809650088.
[I 2026-11-12 17:07:50,886] Trial 2 finished with value: 0.6072753407477924 and parameters: {'n_estimators': 500, 'max_features': 0.619817105976215, 'min_samples_split': 10, 'min_samples_leaf': 7, 'max_depth': 40, 'bootstrap': False}. Best is trial 1 with value: 0.5978150809650088.
[I 2026-11-12 17:11:45,880] Trial 3 finished wi

Optuna: CatBoostRegressor, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-12 18:32:25,455] A new study created in RDB with name: brd4_f081bd1fe370_CatBoostRegressor
[I 2026-11-12 19:25:40,694] Trial 0 finished with value: 0.6130649552662983 and parameters: {'iterations': 500, 'depth': 10, 'learning_rate': 0.08960785365368121, 'l2_leaf_reg': 2.385234757844707, 'random_strength': 0.31203728088487304}. Best is trial 0 with value: 0.6130649552662983.
[I 2026-11-12 19:26:21,267] Trial 1 finished with value: 0.6118395227911959 and parameters: {'iterations': 400, 'depth': 4, 'learning_rate': 0.13394334706750485, 'l2_leaf_reg': 2.416482602989751, 'random_strength': 1.416145155592091}. Best is trial 1 with value: 0.6118395227911959.
[I 2026-11-12 19:45:58,233] Trial 2 finished with value: 0.6234146567794541 and parameters: {'iterations': 300, 'depth': 10, 'learning_rate': 0.12106896936002161, 'l2_leaf_reg': 0.30803400529839686, 'random_strength': 0.36364993441420124}. Best is trial 1 with value: 0.6118395227911959.
[I 2026-11-12 19:47:44,756] Trial 3 finis

Optuna: SVR, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-14 20:10:02,625] A new study created in RDB with name: brd4_f081bd1fe370_SVR


In [7]:
import optuna
db = next(PROJECT_ROOT.rglob("optuna_studies.sqlite3"))
storage = f"sqlite:///{db.resolve().as_posix()}"
for s in optuna.get_all_study_summaries(storage):
    print(s.study_name, "| trials:", s.n_trials)

brd4_56225827313f_ExtraTreesRegressor | trials: 30
brd4_56225827313f_CatBoostRegressor | trials: 30
brd4_56225827313f_SVR | trials: 1


In [8]:
import hashlib, json
from pathlib import Path
import optuna
from src import optuna_tuning as ot
from src.utils import OUTPUTS, load_config, sha256

cfg = load_config()
fp = OUTPUTS / "05_feature_selection/tables/train_selected_features.parquet"
tp = OUTPUTS / "05_feature_selection/tables/train_target.parquet"
sp = OUTPUTS / "05_feature_selection/tables/selected_features.csv"
ip = OUTPUTS / "04_train_test_split/tables/train_ids.csv"

material = {
    "features": sha256(fp),
    "selected_feature_list": sha256(sp),
    "target": sha256(tp),
    "train_ids": sha256(ip),
    "tuner_script": sha256(Path(ot.__file__)),
    "feature_budget": ot.FEATURE_BUDGET,
    "feature_selection": "fixed ordered Stage 05 selected 250-feature list; no fold-specific cleaning or reranking",
    "folds": 5,
    "models": ot.MODELS,
    "tuned_models": ot.TUNED_MODELS,
    "seed": int(cfg.get("seed", 42)),
}
run_id = hashlib.sha256(json.dumps(material, sort_keys=True).encode()).hexdigest()[:12]
print("hash the script will use:", run_id)

db = next(PROJECT_ROOT.rglob("optuna_studies.sqlite3"))
storage = f"sqlite:///{db.resolve().as_posix()}"
s = optuna.load_study(study_name=f"brd4_{run_id}_SVR", storage=storage) \
    if any(x.study_name == f"brd4_{run_id}_SVR" for x in optuna.get_all_study_summaries(storage)) else None
if s is None:
    print("No SVR study with this hash in the database")
else:
    print(s.trials_dataframe()[["number", "state", "value"]])

hash the script will use: f081bd1fe370
No SVR study with this hash in the database


In [9]:
import optuna
from src.utils import stage_dir

d = stage_dir("07_optuna_tuning")
used = (d / "models/optuna_studies.sqlite3").resolve()
print("run() uses:", used, "| exists:", used.exists(), "\n")

for p in PROJECT_ROOT.rglob("*.sqlite3"):
    p = p.resolve()
    print(p, "<-- used by run()" if p == used else "")
    try:
        for s in optuna.get_all_study_summaries(f"sqlite:///{p.as_posix()}"):
            print("   ", s.study_name, "| trials:", s.n_trials)
    except Exception as e:
        print("    could not read:", e)
    print()

run() uses: C:\Users\USER\Documents\E\Code\outputs\07_optuna_tuning\models\optuna_studies.sqlite3 | exists: True 

C:\Users\USER\Documents\E\Code\copy\outputs\07_optuna_tuning\models\optuna_studies.sqlite3 
    brd4_56225827313f_ExtraTreesRegressor | trials: 30
    brd4_56225827313f_CatBoostRegressor | trials: 30
    brd4_56225827313f_SVR | trials: 1

C:\Users\USER\Documents\E\Code\outputs\07_optuna_tuning\models\optuna_studies.sqlite3 <-- used by run()
    brd4_f081bd1fe370_ExtraTreesRegressor | trials: 30
    brd4_f081bd1fe370_CatBoostRegressor | trials: 30
    brd4_f081bd1fe370_SVR | trials: 1



In [ ]:
from src.optuna_tuning import run
summary = run()

Optuna: ExtraTreesRegressor, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-15 19:13:45,449] Using an existing study with name 'brd4_f081bd1fe370_ExtraTreesRegressor' instead of creating a new one.


Optuna: CatBoostRegressor, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-15 19:13:45,689] Using an existing study with name 'brd4_f081bd1fe370_CatBoostRegressor' instead of creating a new one.


Optuna: SVR, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-15 19:13:45,989] Using an existing study with name 'brd4_f081bd1fe370_SVR' instead of creating a new one.
[I 2026-11-16 11:24:36,755] Trial 1 finished with value: 0.6857915198390874 and parameters: {'kernel': 'linear', 'C': 45.70563099801453, 'epsilon': 0.04128205343826222}. Best is trial 1 with value: 0.6857915198390874.
[I 2026-11-16 11:25:00,188] Trial 2 finished with value: 1.1372042874882629 and parameters: {'kernel': 'rbf', 'C': 0.0195172246414495, 'epsilon': 0.21766241123453672, 'gamma': 0.02537815508265665}. Best is trial 1 with value: 0.6857915198390874.
[I 2026-11-16 11:25:59,412] Trial 3 finished with value: 0.655272054233244 and parameters: {'kernel': 'rbf', 'C': 707.2114131472224, 'epsilon': 0.1764971584817571, 'gamma': 0.0007068974950624604}. Best is trial 3 with value: 0.655272054233244.
[I 2026-11-16 11:28:00,723] Trial 4 finished with value: 0.6832007044317986 and parameters: {'kernel': 'linear', 'C': 0.3320559103751956, 'epsilon': 0.026079656598095833}. Bes

In [3]:
import optuna
from src.utils import stage_dir

d = stage_dir("07_optuna_tuning")
used = (d / "models/optuna_studies.sqlite3").resolve()
print("run() uses:", used, "| exists:", used.exists(), "\n")

for p in PROJECT_ROOT.rglob("*.sqlite3"):
    p = p.resolve()
    print(p, "<-- used by run()" if p == used else "")
    try:
        for s in optuna.get_all_study_summaries(f"sqlite:///{p.as_posix()}"):
            print("   ", s.study_name, "| trials:", s.n_trials)
    except Exception as e:
        print("    could not read:", e)
    print()

run() uses: C:\Users\USER\Documents\E\Code\outputs\07_optuna_tuning\models\optuna_studies.sqlite3 | exists: True 

C:\Users\USER\Documents\E\Code\copy\outputs\07_optuna_tuning\models\optuna_studies.sqlite3 
    brd4_56225827313f_ExtraTreesRegressor | trials: 30
    brd4_56225827313f_CatBoostRegressor | trials: 30
    brd4_56225827313f_SVR | trials: 1

C:\Users\USER\Documents\E\Code\outputs\07_optuna_tuning\models\optuna_studies.sqlite3 <-- used by run()
    brd4_f081bd1fe370_ExtraTreesRegressor | trials: 30
    brd4_f081bd1fe370_CatBoostRegressor | trials: 30
    brd4_f081bd1fe370_SVR | trials: 32



In [2]:
import optuna
from collections import Counter

storage = r"sqlite:///C:/Users/USER/Documents/E/Code/outputs/07_optuna_tuning/models/optuna_studies.sqlite3"
for name in optuna.study.get_all_study_names(storage):
    s = optuna.load_study(study_name=name, storage=storage)
    print(name, Counter(t.state.name for t in s.trials))

brd4_f081bd1fe370_ExtraTreesRegressor Counter({'COMPLETE': 30})
brd4_f081bd1fe370_CatBoostRegressor Counter({'COMPLETE': 30})
brd4_f081bd1fe370_SVR Counter({'COMPLETE': 29, 'RUNNING': 4})


In [3]:
import optuna
from optuna.trial import TrialState

storage = r"sqlite:///C:/Users/USER/Documents/E/Code/outputs/07_optuna_tuning/models/optuna_studies.sqlite3"
study = optuna.load_study(study_name="brd4_f081bd1fe370_SVR", storage=storage)

for t in study.trials:
    if t.state == TrialState.RUNNING:
        study._storage.set_trial_state_values(t._trial_id, TrialState.FAIL)

from collections import Counter
print(Counter(t.state.name for t in study.trials))

Counter({'COMPLETE': 29, 'FAIL': 4})


In [4]:
from src.optuna_tuning import run
summary = run()

Optuna: ExtraTreesRegressor, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-18 21:16:35,352] Using an existing study with name 'brd4_f081bd1fe370_ExtraTreesRegressor' instead of creating a new one.


Optuna: CatBoostRegressor, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-18 21:16:35,516] Using an existing study with name 'brd4_f081bd1fe370_CatBoostRegressor' instead of creating a new one.


Optuna: SVR, fixed 250 features, 5-fold GroupKFold, up to 30 trials


[I 2026-11-18 21:16:35,760] Using an existing study with name 'brd4_f081bd1fe370_SVR' instead of creating a new one.
[I 2026-11-19 01:56:48,156] Trial 33 finished with value: 0.683217782191712 and parameters: {'kernel': 'linear', 'C': 40.83317679128585, 'epsilon': 0.059019659476372166}. Best is trial 22 with value: 0.5886360630392922.



Optuna tuning summary (training-only grouped CV; fixed Stage 05 250 features):
              model  best_cv_rmse       parameter_source  trial_number                                                                                                                                            params  cv_mean_fold_rmse  cv_mean_fold_mse  cv_mean_fold_mae  cv_mean_fold_r2  cv_sd_fold_rmse  cv_sd_fold_mse  cv_sd_fold_mae  cv_sd_fold_r2  cv_pooled_oof_rmse  cv_pooled_oof_mse  cv_pooled_oof_mae  cv_pooled_oof_r2
                SVR      0.588636 Optuna best parameters            22                                      {"C": 18.684324183616088, "epsilon": 0.0020663904525143905, "gamma": 0.0008251796314296465, "kernel": "rbf"}           0.588636          0.347903          0.415455         0.754101         0.041992        0.049049        0.028367       0.048894            0.585232           0.342496           0.413342          0.765849
ExtraTreesRegressor      0.589988 Optuna best parameters      

In [5]:
import json
import pandas as pd
from pathlib import Path

best_params_path = PROJECT_ROOT / "outputs/07_optuna_tuning/tables/optuna_best_params.json"
if not best_params_path.is_file():
    raise FileNotFoundError(f"Run the tuning cell first; missing: {best_params_path}")

best_params_report = json.loads(best_params_path.read_text(encoding="utf-8"))
metric_columns = [
    "cv_mean_fold_rmse", "cv_sd_fold_rmse",
    "cv_mean_fold_r2", "cv_sd_fold_r2",
    "cv_mean_fold_mae", "cv_sd_fold_mae",
    "cv_mean_fold_mse", "cv_sd_fold_mse",
    "cv_pooled_oof_rmse", "cv_pooled_oof_r2",
    "cv_pooled_oof_mae", "cv_pooled_oof_mse",
]
report_rows = []
for model_name, result in best_params_report["params_by_model"].items():
    report_rows.append({"model": model_name, **{k: result[k] for k in metric_columns}})
display(pd.DataFrame(report_rows).round(4))

,model,cv_mean_fold_rmse,cv_sd_fold_rmse,cv_mean_fold_r2,cv_sd_fold_r2,cv_mean_fold_mae,cv_sd_fold_mae,cv_mean_fold_mse,cv_sd_fold_mse,cv_pooled_oof_rmse,cv_pooled_oof_r2,cv_pooled_oof_mae,cv_pooled_oof_mse
0,ExtraTreesRegressor,0.5900,0.0418,0.7516,0.0558,0.4246,0.0360,0.3495,0.0487,0.5865,0.7648,0.4214,0.3440
1,CatBoostRegressor,0.5999,0.0420,0.7432,0.0578,0.4368,0.0358,0.3613,0.0498,0.5966,0.7567,0.4338,0.3559
2,SVR,0.5886,0.0420,0.7541,0.0489,0.4155,0.0284,0.3479,0.0490,0.5852,0.7658,0.4133,0.3425
